# 90 - Predict de clusters TF-IDF + HDBSCAN

Este notebook carrega um modelo exportado pelo notebook 74, recebe uma descricao e indica um cluster existente junto de 10 descricoes daquele cluster.

A implementacao do HDBSCAN do scikit-learn nao fornece `predict` para novos itens. Por isso, a atribuicao usa os 10 vizinhos de maior similaridade cosseno entre os exemplos de treino que receberam cluster (exclui o ruido `-1`). A votacao e ponderada pela similaridade. Depois, o notebook mostra os 10 exemplos mais proximos dentro do cluster vencedor. O resultado e uma aproximacao baseada nos assignments existentes; nao e uma nova execucao do HDBSCAN.


In [39]:
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics.pairwise import cosine_similarity

from nofis_classifier.text import normalize_description

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook90_predict")
pd.set_option("display.max_colwidth", 140)


## 1. Selecione o modelo e a descricao

`MODEL_DIRECTORY` e o nome da pasta criada pelo notebook 74 em `models/tfidf_hdbscan/` ou `models/tfidf_mpnet_hdbscan/`. Troque esse valor para consultar outro modelo. `DESCRIPTION_TO_PREDICT` recebe a descricao em texto livre; o mesmo normalizador usado pelo projeto e aplicado antes do TF-IDF.


In [40]:
DESCRIPTION_TO_PREDICT = "Sabonete em barra"
DESCRIPTION_TO_PREDICT = "Pao de forma"
DESCRIPTION_TO_PREDICT = "sabt johnsons roma 80g"

In [41]:

MODEL_DIRECTORY = "202501-202512_20261018T031021Z417600_seed42_char_wb_3_5_alpha0.5_min_cluster_size20_min_samples10_eom"

VOTING_NEIGHBORS = 10
EXAMPLES_TO_SHOW = 10

MODEL_RELATIVE_PATHS = [
    Path("models/tfidf_hdbscan") / MODEL_DIRECTORY,
    Path("models/tfidf_mpnet_hdbscan") / MODEL_DIRECTORY,
]
PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "processed").is_dir()),
    None,
)
MODEL_PATH_CANDIDATES = list(MODEL_RELATIVE_PATHS)
if PROJECT_ROOT is not None:
    for relative_path in MODEL_RELATIVE_PATHS:
        MODEL_PATH_CANDIDATES.extend([
            PROJECT_ROOT / relative_path,
            PROJECT_ROOT / "notebooks" / relative_path,
        ])
MODEL_DIR = next((path for path in MODEL_PATH_CANDIDATES if path.is_dir()), None)
if MODEL_DIR is None:
    raise FileNotFoundError(
        "Pasta do modelo nao encontrada. Verifique MODEL_DIRECTORY. Caminhos verificados: "
        f"{MODEL_PATH_CANDIDATES}"
    )


## 2. Carregue os artefatos

O modelo e carregado para conferir o Run ID e validar que seus labels correspondem ao arquivo de assignments. O vetorizador transforma tanto os exemplos de treino quanto a descricao nova com o vocabulario salvo.


In [42]:
CONFIG_PATH = MODEL_DIR / "config.json"
VECTORIZER_PATH = MODEL_DIR / "tfidf_vectorizer.pkl"
MODEL_FILE_PATH = MODEL_DIR / "hdbscan_model.pkl"
ASSIGNMENTS_PATH = MODEL_DIR / "training_assignments.csv"

for required_path in [CONFIG_PATH, VECTORIZER_PATH, MODEL_FILE_PATH, ASSIGNMENTS_PATH]:
    if not required_path.is_file():
        raise FileNotFoundError(f"Artefato ausente: {required_path}")

model_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
vectorizer = pd.read_pickle(VECTORIZER_PATH)
hdbscan_model = pd.read_pickle(MODEL_FILE_PATH)
training_assignments = pd.read_csv(ASSIGNMENTS_PATH, encoding="utf-8-sig")

required_columns = {"descricao_normalizada", "frequencia", "cluster"}
missing_columns = required_columns - set(training_assignments.columns)
if missing_columns:
    raise ValueError(f"Colunas ausentes nos assignments: {sorted(missing_columns)}")
if len(hdbscan_model.labels_) != len(training_assignments):
    raise ValueError("O modelo e os assignments tem quantidades diferentes de descricoes.")
if not np.array_equal(hdbscan_model.labels_, training_assignments["cluster"].to_numpy()):
    raise ValueError("Os labels do modelo nao correspondem aos assignments salvos.")

logger.info("Modelo carregado: %s", model_config["run_id"])
logger.info("Pasta do modelo: %s", MODEL_DIR)
logger.info("Descricoes de treino: %d", len(training_assignments))


2026-10-18 10:21:40,173 INFO Modelo carregado: seed42_char_wb_3_5_alpha0.5_min_cluster_size20_min_samples10_eom
2026-10-18 10:21:40,173 INFO Pasta do modelo: c:\Users\samue\Scripts\Nofis-Classifier\models\tfidf_mpnet_hdbscan\202501-202512_20261018T031021Z417600_seed42_char_wb_3_5_alpha0.5_min_cluster_size20_min_samples10_eom
2026-10-18 10:21:40,174 INFO Descricoes de treino: 10000


## 3. Consulte o cluster e os exemplos

A votacao considera os vizinhos nao classificados como ruido. O cluster vencedor recebe o maior total de similaridade entre os vizinhos consultados. A tabela final lista os exemplos mais similares dentro desse cluster; se houver menos de 10 exemplos, todos serao exibidos.


In [43]:
if VOTING_NEIGHBORS < 1 or EXAMPLES_TO_SHOW < 1:
    raise ValueError("VOTING_NEIGHBORS e EXAMPLES_TO_SHOW devem ser positivos.")

normalized_description = normalize_description(DESCRIPTION_TO_PREDICT)
if not normalized_description:
    raise ValueError("Informe uma descricao nao vazia em DESCRIPTION_TO_PREDICT.")

training_texts = training_assignments["descricao_normalizada"].fillna("").astype(str)
training_matrix = vectorizer.transform(training_texts)
query_matrix = vectorizer.transform([normalized_description])
if query_matrix.nnz == 0:
    raise ValueError(
        "A descricao nao contem termos presentes no vocabulario TF-IDF deste modelo. "
        "Tente uma descricao com mais palavras ou consulte outro modelo."
    )

cluster_labels = training_assignments["cluster"].to_numpy(dtype=int)
has_cluster = cluster_labels != -1
has_features = np.asarray(training_matrix.getnnz(axis=1)).ravel() > 0
candidate_positions = np.flatnonzero(has_cluster & has_features)
if candidate_positions.size == 0:
    raise ValueError("O modelo nao tem exemplos vetorizados em clusters validos.")

candidate_similarities = cosine_similarity(
    query_matrix, training_matrix[candidate_positions]
).ravel()
neighbor_count = min(VOTING_NEIGHBORS, len(candidate_positions))
neighbor_order = np.argsort(-candidate_similarities, kind="stable")[:neighbor_count]
neighbor_positions = candidate_positions[neighbor_order]
neighbor_similarities = candidate_similarities[neighbor_order]
neighbor_labels = cluster_labels[neighbor_positions]

votes = (
    pd.DataFrame({"cluster": neighbor_labels, "similaridade": neighbor_similarities})
    .groupby("cluster", as_index=False)
    .agg(votos=("cluster", "size"), similaridade_total=("similaridade", "sum"),
         melhor_similaridade=("similaridade", "max"))
    .sort_values(
        ["similaridade_total", "votos", "melhor_similaridade", "cluster"],
        ascending=[False, False, False, True],
    )
    .reset_index(drop=True)
)

if votes.empty or votes.iloc[0]["similaridade_total"] <= 0:
    raise ValueError(
        "Nao houve similaridade positiva com exemplos de clusters existentes; "
        "o notebook nao vai atribuir um cluster sem correspondencia textual."
    )

predicted_cluster = int(votes.iloc[0]["cluster"])
cluster_positions = np.flatnonzero(cluster_labels == predicted_cluster)
cluster_similarities = cosine_similarity(
    query_matrix, training_matrix[cluster_positions]
).ravel()
example_order = np.argsort(-cluster_similarities, kind="stable")[:EXAMPLES_TO_SHOW]
example_positions = cluster_positions[example_order]

examples = training_assignments.iloc[example_positions].copy().reset_index(drop=True)
examples.insert(0, "ordem_similaridade", np.arange(1, len(examples) + 1))
examples["similaridade_cosseno"] = cluster_similarities[example_order]

prediction_summary = pd.DataFrame([{
    "descricao_informada": DESCRIPTION_TO_PREDICT,
    "descricao_normalizada": normalized_description,
    "cluster_sugerido": predicted_cluster,
    "vizinhos_usados_na_votacao": neighbor_count,
    "votos_no_cluster": int(votes.iloc[0]["votos"]),
    "similaridade_maxima": float(votes.iloc[0]["melhor_similaridade"]),
}])

display(prediction_summary)
display(votes)
display(examples)


,descricao_informada,descricao_normalizada,cluster_sugerido,vizinhos_usados_na_votacao,votos_no_cluster,similaridade_maxima
0,sabt johnsons roma 80g,sabt johnsons roma 80g,0,10,10,0.43781


,cluster,votos,similaridade_total,melhor_similaridade
0,0,10,2.292178,0.43781


,ordem_similaridade,descricao_id,descricao_normalizada,frequencia,cluster,ruido,similaridade_cosseno
0,1,8495,sab johnsons lavanda,5,0,False,0.437810
1,2,940,aroma natural de fumaca aroma em po,2,0,False,0.252957
2,3,7034,pano de prato bainha roma,2,0,False,0.239322
3,4,2601,choc lacta 80g laka,9,0,False,0.216292
4,5,6407,mini tortinha 80g,1,0,False,0.200204
5,6,2613,choc neugebauer ao leite 80g,1,0,False,0.196579
6,7,1329,bar choc lacta amaro 80g,3,0,False,0.194198
7,8,1112,azeitona rivoli 80g,1,0,False,0.193364
8,9,2612,choc neugebauer 80g ao leite 18,2,0,False,0.187904
9,10,1334,barra choc lacta 80g ao leite,1,0,False,0.173548
